# Arena rating convergence

Toy simulator: a fresh account at rating 1000 fights opponents drawn
from a normal distribution centered on the player's true skill. Plots
the rating trajectory across N matches.

Used to check whether our `RATING_WIN=25` / `RATING_LOSS=-15` constants
let players converge in a reasonable timeframe (we'd expect ~30-50 matches).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

RATING_WIN = 25
RATING_LOSS = -15
MIN_RATING = 0

def simulate_player(true_skill: int, n_matches: int = 100, start_rating: int = 1000) -> list[int]:
    rng = random.Random()
    rating = start_rating
    history = [rating]
    for _ in range(n_matches):
        opp_rating = rng.gauss(rating, 100)  # matchmaking centered on player
        # Win probability rises sharply with skill margin.
        win_p = 1 / (1 + 10 ** ((opp_rating - true_skill) / 200))
        if rng.random() < win_p:
            rating = max(MIN_RATING, rating + RATING_WIN)
        else:
            rating = max(MIN_RATING, rating + RATING_LOSS)
        history.append(rating)
    return history

In [ ]:
# Run 4 different true-skill bands across many matches.
fig, ax = plt.subplots(figsize=(11, 5))
for true_skill, color in [(1500, '#ffd86b'), (1200, '#6dd39a'), (1000, '#59a0ff'), (700, '#c97aff')]:
    runs = np.array([simulate_player(true_skill, 80) for _ in range(50)])
    mean = runs.mean(axis=0)
    ax.plot(mean, color=color, label=f'true skill {true_skill}', linewidth=2)
    ax.axhline(true_skill, color=color, linestyle='--', alpha=0.3)
ax.set_xlabel('matches played'); ax.set_ylabel('rating (mean of 50 runs)')
ax.set_title('Arena rating convergence — fresh account starts at 1000')
ax.legend()
fig.tight_layout()
fig.savefig('output/arena_convergence.png', dpi=120, facecolor='white')
plt.show()